In [1]:
import math
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPolygon

import folium
from folium import Choropleth, Marker
from folium.plugins import HeatMap, MarkerCluster

from pathlib import Path
DATASETS = Path('../datasets/geospatial-learn-course-data')

In [2]:
def embed_map(m, file_name):
    from IPython.display import IFrame
    m.save(file_name)
    return IFrame(file_name, width='100%', height='500px')

In [3]:
COLLISIONS_DATA = DATASETS / "NYPD_Motor_Vehicle_Collisions/NYPD_Motor_Vehicle_Collisions.shp"

collisions = gpd.read_file(COLLISIONS_DATA)
collisions.head()

,DATE,TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET,CROSS STRE,OFF STREET,...,CONTRIBU_2,CONTRIBU_3,CONTRIBU_4,UNIQUE KEY,VEHICLE TY,VEHICLE _1,VEHICLE _2,VEHICLE _3,VEHICLE _4,geometry
0,07/30/2019,0:00,BRONX,10464,40.841100,-73.784960,"(40.8411, -73.78496)",None,None,121 PILOT STREET,...,Unspecified,None,None,4180045,Sedan,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,None,None,POINT (1043750.211 245785.815)
1,07/30/2019,0:10,QUEENS,11423,40.710827,-73.770660,"(40.710827, -73.77066)",JAMAICA AVENUE,188 STREET,None,...,None,None,None,4180007,Sedan,Sedan,None,None,None,POINT (1047831.185 198333.171)
2,07/30/2019,0:25,None,None,40.880318,-73.841286,"(40.880318, -73.841286)",BOSTON ROAD,None,None,...,None,None,None,4179575,Sedan,Station Wagon/Sport Utility Vehicle,None,None,None,POINT (1028139.293 260041.178)
3,07/30/2019,0:35,MANHATTAN,10036,40.756744,-73.984590,"(40.756744, -73.98459)",None,None,155 WEST 44 STREET,...,None,None,None,4179544,Box Truck,Station Wagon/Sport Utility Vehicle,None,None,None,POINT (988519.261 214979.32)
4,07/30/2019,10:00,BROOKLYN,11223,40.600090,-73.965910,"(40.60009, -73.96591)",AVENUE T,OCEAN PARKWAY,None,...,None,None,None,4180660,Station Wagon/Sport Utility Vehicle,Bike,None,None,None,POINT (993716.669 157907.212)


In [4]:
# Show the map
LATITUDE, LONGITUDE = 'LATITUDE', 'LONGITUDE'
NEW_YORK = [40.7, -74]
START_ZOOM = 10

m_1 = folium.Map(location=NEW_YORK, zoom_start=START_ZOOM) 

# Your code here: Visualize the collision data
HeatMap(data=collisions[[LATITUDE, LONGITUDE]], radius=9).add_to(m_1)


embed_map(m_1, "pa_q1.html")

In [5]:
HOSPITALS_DATA = DATASETS / "nyu_2451_34494/nyu_2451_34494.shp"
hospitals = gpd.read_file(HOSPITALS_DATA)
hospitals.head()

,id,name,address,zip,factype,facname,capacity,capname,bcode,xcoord,ycoord,latitude,longitude,geometry
0,317000001H1178,BRONX-LEBANON HOSPITAL CENTER - CONCOURSE DIVI...,1650 Grand Concourse,10457,3102,Hospital,415,Beds,36005,1008872.0,246596.0,40.843490,-73.911010,POINT (1008872 246596)
1,317000001H1164,BRONX-LEBANON HOSPITAL CENTER - FULTON DIVISION,1276 Fulton Ave,10456,3102,Hospital,164,Beds,36005,1011044.0,242204.0,40.831429,-73.903178,POINT (1011044 242204)
2,317000011H1175,CALVARY HOSPITAL INC,1740-70 Eastchester Rd,10461,3102,Hospital,225,Beds,36005,1027505.0,248287.0,40.848060,-73.843656,POINT (1027505 248287)
3,317000002H1165,JACOBI MEDICAL CENTER,1400 Pelham Pkwy,10461,3102,Hospital,457,Beds,36005,1027042.0,251065.0,40.855687,-73.845311,POINT (1027042 251065)
4,317000008H1172,LINCOLN MEDICAL & MENTAL HEALTH CENTER,234 E 149 St,10451,3102,Hospital,362,Beds,36005,1005154.0,236853.0,40.816758,-73.924478,POINT (1005154 236853)


In [6]:
# Visualize the hospital locations.
m_2 = folium.Map(location=NEW_YORK, zoom_start=START_ZOOM) 
mc = MarkerCluster()

for idx, row in hospitals.iterrows():
    mc.add_child(Marker([row[LATITUDE.lower()], row[LONGITUDE.lower()]]))                 
m_2.add_child(mc)

embed_map(m_2, "pa_q2.html")

In [7]:
coverage = gpd.GeoDataFrame(geometry=hospitals.geometry).buffer(1e4)
my_union = coverage.geometry.union_all()
outside_range = collisions.loc[~collisions["geometry"].apply(lambda x: my_union.contains(x))]

percentage = round(100*len(outside_range)/len(collisions), 2)
print(f"Percentage of collisions more than 10 km away from the closest hospital: {percentage}%")

Percentage of collisions more than 10 km away from the closest hospital: 15.12%


In [8]:
"""
When collisions occur in distant locations, it becomes even more vital that injured persons are 
transported to the nearest available hospital.

With this in mind, you decide to create a recommender that:

- takes the location of the crash (in EPSG 2263) as input,
- finds the closest hospital (where distance calculations are done in EPSG 2263), and
- returns the name of the closest hospital.
"""
def best_hospital(collision_location):
    closest_hospital_index = hospitals.geometry.distance(collision_location).idxmin()
    closest_hospital = hospitals.iloc[closest_hospital_index]
    return closest_hospital['name']

# This should suggest CALVARY HOSPITAL INC
print(best_hospital(outside_range.geometry.iloc[0]))

CALVARY HOSPITAL INC


In [9]:
# Considering only collisions in the outside_range DataFrame, which hospital is most recommended?
highest_demand = outside_range.geometry.apply(best_hospital).value_counts().idxmax()
highest_demand                                     

'JAMAICA HOSPITAL MEDICAL CENTER'

In [10]:
# visualize hospital locations, in addition to collisions that occurred more than 
# 10 kilometers away from the closest hospital.
m_6 = folium.Map(location=NEW_YORK, zoom_start=START_ZOOM) 

coverage = gpd.GeoDataFrame(geometry=hospitals.geometry).buffer(1e4)
folium.GeoJson(coverage.geometry.to_crs(epsg=4326)).add_to(m_6)
HeatMap(data=outside_range[[LATITUDE, LONGITUDE]], radius=9).add_to(m_6)
folium.LatLngPopup().add_to(m_6)

embed_map(m_6, 'pa_m6.html')

In [13]:
"""
Click anywhere on the map to see a pop-up with the corresponding location in latitude and longitude.

The city of New York must decide locations for two brand new hospitals. They specifically want to 
identifying locations to bring the calculated percentage from step 3) to less than ten percent. 
Using the map (and without worrying about zoning laws or what potential buildings would have to be 
removed in order to build the hospitals), identify two locations that would help the city 
accomplish this goal.

Put the proposed latitude and longitude for hospital 1 in lat_1 and long_1, respectively. 
(Likewise for hospital 2.)
"""

# Proposed location of hospital 1
lat_1 = 40.6714
long_1 = -73.8492

# Proposed location of hospital 2
lat_2 = 40.6702
long_2 = -73.7612

In [16]:
EPSG_4326 = 'epsg:4326'

new_df = pd.DataFrame({
    LATITUDE: [lat_1, lat_2],
    LONGITUDE: [long_1, long_2]
})

new_gdf = gpd.GeoDataFrame(new_df, crs=EPSG_4326,
                           geometry=gpd.points_from_xy(new_df[LONGITUDE], new_df[LATITUDE]))
new_gdf = new_gdf.to_crs(epsg=2263)

# get new percentage
new_coverage = gpd.GeoDataFrame(geometry=new_gdf.geometry).buffer(1e4)
new_my_union = new_coverage.geometry.union_all()
new_outside_range = outside_range.loc[~outside_range["geometry"].apply(lambda x: new_my_union.contains(x))]
new_percentage = round(100*len(new_outside_range)/len(collisions), 2)
print(f"(NEW) Percentage of collisions more than 10 km away from the closest hospital: {new_percentage}%")
    
# make the map
m = folium.Map(location=NEW_YORK, zoom_start=START_ZOOM) 
folium.GeoJson(coverage.geometry.to_crs(epsg=4326)).add_to(m)
folium.GeoJson(new_coverage.geometry.to_crs(epsg=4326)).add_to(m)
for idx, row in new_gdf.iterrows():
    Marker([row[LATITUDE], row[LONGITUDE]]).add_to(m)
HeatMap(data=new_outside_range[[LATITUDE, LONGITUDE]], radius=9).add_to(m)

folium.LatLngPopup().add_to(m)
display(embed_map(m, 'q_6.html'))


(NEW) Percentage of collisions more than 10 km away from the closest hospital: 9.12%
